# 1、ChatMessageHistory的使用

① 实例的创建，方法的调用


In [1]:
from langchain.memory import ChatMessageHistory

history = ChatMessageHistory()

#添加消息的第1种方式
history.add_user_message("你好，我是小明")
history.add_ai_message("很高兴认识你，我是人工智能助手")
history.add_user_message("帮我解释一下1 + 2 * 3 = ？")

print(history.messages)  #返回消息构成的列表

[HumanMessage(content='你好，我是小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你，我是人工智能助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我解释一下1 + 2 * 3 = ？', additional_kwargs={}, response_metadata={})]


② 调用LLM

In [2]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
import os
import dotenv
from langchain.memory import ChatMessageHistory

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

# 1、获取大模型
chat_model = ChatOpenAI(
    model="gpt-4o-mini"
)

#
history = ChatMessageHistory()
#添加消息的第2种方式
history.add_user_message(HumanMessage(content="你好，我是小明"))
history.add_ai_message(AIMessage(content="很高兴认识你，我是人工智能助手"))
history.add_user_message(HumanMessage(content="帮我解释一下1 + 2 * 3 = ？"))

print(history.messages)

# chat_model.invoke("你是什么模型？")
#
chat_model.invoke(history.messages)

[HumanMessage(content='你好，我是小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你，我是人工智能助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我解释一下1 + 2 * 3 = ？', additional_kwargs={}, response_metadata={})]


AIMessage(content='在数学中，运算的顺序非常重要。根据优先级规则，乘法运算的优先级高于加法运算。因此，在计算表达式 \\(1 + 2 * 3\\) 时，我们首先进行乘法运算。\n\n1. 先计算 \\(2 * 3\\)，结果为 6。\n2. 然后将这个结果与 1 相加：\\(1 + 6 = 7\\)。\n\n所以， \\(1 + 2 * 3 = 7\\)。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 44, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'finish_reason': 'stop', 'logprobs': None}, id='run-9cc5f378-ef83-4d44-935e-39c78bd64e13-0', usage_metadata={'input_tokens': 44, 'output_tokens': 113, 'total_tokens': 157, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# 2、 ConversationBufferMemory的使用

① ConversationBufferMemory的基本使用，使用其主要的api

In [3]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()

#添加信息
memory.save_context(
    inputs={"input": "我是人类，我叫小明"},  #对应着人类的输入信息,即HumanMessage
    outputs={"output": "我是ai助手，我叫小智"},  #对应着ai的输出信息,即AIMessage
)
memory.save_context(
    inputs={"input": "你好，帮我计算一下1 + 2 * 3 = ？"},  #对应着人类的输入信息
    outputs={"output": "结果是 7 "},  #对应着ai的输出信息
)

#两种查看结果的方式
#第1种：
# print(memory.chat_memory.messages)

#第2种：此时此字典的方式返回memory种存储的信息。其默认的key是：history!
memory.load_memory_variables({})

C:\Users\41507\AppData\Local\Temp\ipykernel_11580\2789062009.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


{'history': 'Human: 我是人类，我叫小明\nAI: 我是ai助手，我叫小智\nHuman: 你好，帮我计算一下1 + 2 * 3 = ？\nAI: 结果是 7 '}

体会内部的属性：return_messages

In [4]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    return_messages=True  #默认值是False
)

#添加信息
memory.save_context(
    inputs={"input": "我是人类，我叫小明"},  #对应着人类的输入信息,即HumanMessage
    outputs={"output": "我是ai助手，我叫小智"},  #对应着ai的输出信息,即AIMessage
)
memory.save_context(
    inputs={"input": "你好，帮我计算一下1 + 2 * 3 = ？"},  #对应着人类的输入信息
    outputs={"output": "结果是 7 "},  #对应着ai的输出信息
)

#此时此字典的方式返回memory种存储的信息。其默认的key是：history!
memory.load_memory_variables({})

{'history': [HumanMessage(content='我是人类，我叫小明', additional_kwargs={}, response_metadata={}),
  AIMessage(content='我是ai助手，我叫小智', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='你好，帮我计算一下1 + 2 * 3 = ？', additional_kwargs={}, response_metadata={}),
  AIMessage(content='结果是 7 ', additional_kwargs={}, response_metadata={})]}

② 结合LLM,使用PromptTemplate结构,对应着return_messages = False的情况

In [5]:
from langchain.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate, MessagesPlaceholder

#1、获取大模型
chat_model = ChatOpenAI(
    model="gpt-4o-mini"
)

#2、提供PromptTemplate结构
template = """
我是一个人工智能的助手，可以详细的回复用户的问题。

历史问题：{history}

当前问题：{question}

"""
prompt_template = PromptTemplate.from_template(template)

#3、提供ConversationBufferMemory的实例
memory = ConversationBufferMemory()

#4、提供LLMChain的实例
llm_chain = LLMChain(
    llm=chat_model,
    prompt=prompt_template,
    memory=memory,
)

#5、调用过程
llm_chain.invoke({"question": "1 + 3 = ？"})


C:\Users\41507\AppData\Local\Temp\ipykernel_11580\2508634722.py:24: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(


{'question': '1 + 3 = ？', 'history': '', 'text': '1 + 3 = 4。'}

In [6]:
llm_chain.invoke({"question": "我刚才问了什么问题？"})

{'question': '我刚才问了什么问题？',
 'history': 'Human: 1 + 3 = ？\nAI: 1 + 3 = 4。',
 'text': '你刚才问了“1 + 3 = ？”这个问题。'}

如上的测试，可以进一步修改为：

In [7]:
from langchain.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate

#1、获取大模型
chat_model = ChatOpenAI(
    model="gpt-4o-mini"
)

#2、提供PromptTemplate结构
template = """
我是一个人工智能的助手，可以详细的回复用户的问题。

历史问题：{chat_history}

当前问题：{question1}

"""
prompt_template = PromptTemplate.from_template(template)

#3、提供ConversationBufferMemory的实例
# ConversationBufferMemory内部的输出结果的key的名称，默认是history。要想
#在PromptTemplate中的历史记录中保存，要与PromptTemplate中保存历史记录的变量名相同
memory = ConversationBufferMemory(memory_key="chat_history")

#4、提供LLMChain的实例
llm_chain = LLMChain(
    llm=chat_model,
    prompt=prompt_template,
    memory=memory,
)

#5、调用过程
llm_chain.invoke({"question1": "1 + 3 = ？"})


{'question1': '1 + 3 = ？',
 'chat_history': '',
 'text': '1 + 3 = 4。 如果你有其他数学问题或者需要进一步的帮助，请随时问我！'}

In [8]:
llm_chain.invoke({"question1": "我刚才问了什么问题？"})

{'question1': '我刚才问了什么问题？',
 'chat_history': 'Human: 1 + 3 = ？\nAI: 1 + 3 = 4。 如果你有其他数学问题或者需要进一步的帮助，请随时问我！',
 'text': '你刚才问的问题是“1 + 3 = ？”而我回答了这个问题，结果是4。如果你有更多的问题或需要进一步的帮助，请告诉我！'}

③ 结合LLM,使用ChatPromptTemplate结构,对应着return_messages = True的情况

In [9]:
from langchain.chains.llm import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import MessagesPlaceholder

#1、获取大模型
chat_model = ChatOpenAI(
    model="gpt-4o-mini"
)

#2、提供ChatPromptTemplate结构
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "我是人工智能助手，可以详细的解决用户问题"),
    MessagesPlaceholder(variable_name="chat_history"),  #要声明在human最后的问题的前面
    ("human", "我的问题是{question}"),
])

#3、提供ConversationBufferMemory的实例
# ConversationBufferMemory内部的输出结果的key的名称，默认是history。要想
#在PromptTemplate中的历史记录中保存，要与PromptTemplate中保存历史记录的变量名相同
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

#4、提供LLMChain的实例
llm_chain = LLMChain(
    llm=chat_model,
    prompt=prompt_template,
    memory=memory,
)

#5、调用过程
llm_chain.invoke({"question": "1 + 3 = ？"})


{'question': '1 + 3 = ？',
 'chat_history': [HumanMessage(content='1 + 3 = ？', additional_kwargs={}, response_metadata={}),
  AIMessage(content='1 + 3 = 4。', additional_kwargs={}, response_metadata={})],
 'text': '1 + 3 = 4。'}

In [10]:
llm_chain.invoke({"question": "我刚才问了什么问题？"})

{'question': '我刚才问了什么问题？',
 'chat_history': [HumanMessage(content='1 + 3 = ？', additional_kwargs={}, response_metadata={}),
  AIMessage(content='1 + 3 = 4。', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='我刚才问了什么问题？', additional_kwargs={}, response_metadata={}),
  AIMessage(content='您刚才问的问题是“1 + 3 = ？”', additional_kwargs={}, response_metadata={})],
 'text': '您刚才问的问题是“1 + 3 = ？”'}